# Finetuning Friend Persona — Qwen2.5-14B-Instruct (Unsloth + TRL, 2-stage)

**Workflow:** Runtime -> Change runtime type -> **A100 GPU**, then run cells sequentially.

* **Target Modules** — Attention-only (`q_proj`, `k_proj`, `v_proj`, `o_proj`) with `r = 32, alpha = 48` (`alpha/r = 1.5`) to preserve base model encyclopedic knowledge and syntactic logic while strongly acquiring persona style.
* Stage 1 — Persona & vocabulary acquisition (`lr = 1.0e-4`, cosine scheduler, full dataset, context length `4096`).
* Stage 2 — Modern punchy-style annealing (`lr = 2e-5`, cosine scheduler, weight decay `0.05`, top 20% deduplicated modern short-reply dialogues, effective batch size `8`, context length `4096`).
* Loss masking — Calculated strictly on assistant responses (`train_on_responses_only`).
* Validation split — 5% of dialogues per friend/chat bucket without pre-split shuffling to prevent temporal weighting and overlap leakage.
* Export — Saves the final HuggingFace PEFT adapter (`lora_final`) and backs it up to Google Drive for local conversion via `tools/convert_lora_to_gguf_local.py`.

In [ ]:
# [Cell 1]: Install dependencies
# If Colab prompts to "Restart session" after installation, restart and continue from Cell 2.
!pip install -q --upgrade pip
!pip install -q unsloth
!pip install -q --upgrade unsloth_zoo
!pip install -q sentencepiece protobuf

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU found. Enable A100 GPU: Runtime -> Change runtime type.")

In [ ]:
# [Cell 2]: Configuration (hyperparameters, paths, inference guardrails)
import os

# ------------- DATA -------------
PERSONA_ID = "friend_1"
DATA_PATH_OVERRIDE = "/content/drive/MyDrive/GVK_dataset_expanded.jsonl"
DROP_LEADING_ASSISTANT = True      # strip leading assistant turns before the first user turn
CHUNK_OVERLAP_EXCHANGES = 3        # number of (user -> assistant) exchanges to overlap when splitting long dialogues
VAL_FRACTION = 0.05                # 5% of dialogues per friend/chat allocated to validation
MIN_CONVS_FOR_VAL = 40             # disable validation split if total dialogues are below this threshold
SEED = 3407

# ------------- MODEL -------------
BASE_MODEL = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096
RESUME_FROM_STAGE1 = False         # True to resume from a saved Stage 1 adapter after session restart

# ------------- LoRA -------------
LORA_R = 32
LORA_ALPHA = 48                    # alpha/r = 1.5 (sweet spot between persona strength and multi-turn attention coherence)
LORA_DROPOUT = 0.00
# Таргетуємо ТІЛЬКИ шари уваги (Attention) для збереження 100% енциклопедичних знань бази:
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
# Альтернатива (всі 7 лінійних шарів, якщо потрібно запікати факти у ваги):
# TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ------------- STAGE 1: Base & Vocabulary -------------
STAGE1_MAX_LEN = 4096
STAGE1_LR = 1.0e-4                   # 1.0e-4 для attention-only (або 5e-5..7e-5 для всіх 7 шарів)
STAGE1_EPOCHS = 1
STAGE1_BATCH = 2
STAGE1_GRAD_ACC = 4
STAGE1_WARMUP_RATIO = 0.03
STAGE1_WEIGHT_DECAY = 0.01
STAGE1_SCHEDULER = "cosine"

# ------------- STAGE 2: Style Annealing -------------
STAGE2_MAX_LEN = 4096
STAGE2_LR = 2e-5
STAGE2_EPOCHS = 1
STAGE2_BATCH = 2
STAGE2_GRAD_ACC = 4
STAGE2_WARMUP_RATIO = 0.05
STAGE2_WEIGHT_DECAY = 0.05
STAGE2_SCHEDULER = "cosine"
STAGE2_TOP_FRACTION = 0.20         # top 20% deduplicated modern short-reply dialogue chunks for Stage 2
STAGE2_MIN_EXAMPLES = 32
STAGE2_MIN_REPLY_CHARS = 8         # sweet spot for punchy Telegram replies (1-3 short lines)
STAGE2_MAX_REPLY_CHARS = 180
STAGE2_MAX_REPLY_LINES = 4

# ------------- INFERENCE GUARDRAILS -------------
GEN_MAX_NEW_TOKENS = 128
GEN_TEMPERATURE = 0.60             # 0.58 ... 0.65
GEN_TOP_P = 0.85                   # 0.85 ... 0.90
GEN_REPETITION_PENALTY = 1.05      # 1.04 ... 1.06
TEST_PROMPTS = [
    "Привіт! Як справи?",
    "Що робиш на вихідних?",
    "го рл",
    "Слухай, можеш пояснити, чим ти взагалі займаєшся на навчанні?",
]

# ------------- EXPORT & BACKUP -------------
COPY_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/qwen_friend"

WORK_DIR = "/content"
STAGE1_OUT = os.path.join(WORK_DIR, "outputs_stage1")
STAGE2_OUT = os.path.join(WORK_DIR, "outputs_stage2")
STAGE1_LORA_DIR = os.path.join(WORK_DIR, "lora_stage1")
FINAL_LORA_DIR = os.path.join(WORK_DIR, "lora_final")

print(f"Configuration saved. Persona: {PERSONA_ID} | MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH} | STAGE1_LR: {STAGE1_LR} | STAGE2_LR: {STAGE2_LR}")

In [ ]:
# [Cell 3]: Mount Google Drive and locate the .jsonl dataset
from google.colab import drive
import glob

if not os.path.ismount("/content/drive"):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

if DATA_PATH_OVERRIDE and os.path.exists(DATA_PATH_OVERRIDE):
    DATA_PATH = DATA_PATH_OVERRIDE
    print(f"Using dataset from Google Drive: {DATA_PATH} ({os.path.getsize(DATA_PATH) / 1024**2:.1f} MB)")
else:
    found = glob.glob("/content/drive/MyDrive/**/GVK_dataset_expanded.jsonl", recursive=True)
    if found:
        DATA_PATH = found[0]
        print(f"Found dataset on Google Drive: {DATA_PATH} ({os.path.getsize(DATA_PATH) / 1024**2:.1f} MB)")
    else:
        from google.colab import files
        print(f"Dataset file not found at {DATA_PATH_OVERRIDE!r} on Google Drive.")
        print("Upload GVK_dataset_expanded.jsonl to Google Drive and re-run this cell, or select the file below:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No dataset file uploaded.")
        DATA_PATH = os.path.join(WORK_DIR, list(uploaded.keys())[0])
        print(f"Uploaded dataset: {DATA_PATH} ({os.path.getsize(DATA_PATH) / 1024**2:.1f} MB)")

In [ ]:
# [Cell 4]: Parse and normalize dialogues (using the system prompt embedded in the dataset)
import json
import re
import random

ROLE_MAP = {
    "user": "user", "human": "user", "prompter": "user", "customer": "user", "client": "user",
    "assistant": "assistant", "gpt": "assistant", "model": "assistant", "bot": "assistant",
    "chatgpt": "assistant", "ai": "assistant",
    "system": "system",
}


def _content_to_text(content):
    """Convert message content into a plain string."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict) and part.get("type", "text") == "text" and "text" in part:
                parts.append(str(part["text"]))
        return "\n".join(parts)
    if isinstance(content, dict) and "text" in content:
        return str(content["text"])
    return str(content)


def _extract_raw_messages(record):
    """Extract the list of messages from a JSON record."""
    if isinstance(record, list):
        return record
    if isinstance(record, dict):
        for key in ("messages", "conversations", "conversation", "dialog", "dialogue", "chat"):
            if isinstance(record.get(key), list):
                return record[key]
    return None


def _extract_quality(record):
    if isinstance(record, dict):
        for key in ("quality", "score", "rating"):
            if key in record:
                try:
                    return float(record[key])
                except (TypeError, ValueError):
                    return None
    return None


def _extract_friend_key(record, system_prompt):
    """Identify the interlocutor/group bucket from the record or its system prompt suffix."""
    if isinstance(record, dict) and record.get("interlocutor"):
        return str(record["interlocutor"]).strip()
    if system_prompt:
        blocks = [b.strip() for b in system_prompt.strip().split("\n\n") if b.strip()]
        if blocks:
            return blocks[-1]
    return "default"


def normalize_messages(raw_messages):
    """Preserve the system prompt from the dataset, normalize roles, and merge consecutive turns."""
    system_parts = []
    msgs = []
    for m in raw_messages:
        if not isinstance(m, dict):
            continue
        role = m.get("role", m.get("from", m.get("speaker")))
        role = ROLE_MAP.get(str(role).strip().lower()) if role is not None else None
        if role is None:
            continue
        text = _content_to_text(m.get("content", m.get("value", m.get("text")))).strip()
        if not text:
            continue
        if role == "system":
            system_parts.append(text)
            continue
        if msgs and msgs[-1]["role"] == role:
            msgs[-1]["content"] += "\n" + text
        else:
            msgs.append({"role": role, "content": text})
    if DROP_LEADING_ASSISTANT:
        while msgs and msgs[0]["role"] == "assistant":
            msgs.pop(0)
    while msgs and msgs[-1]["role"] == "user":
        msgs.pop()
    has_user = any(m["role"] == "user" for m in msgs)
    has_assistant = any(m["role"] == "assistant" for m in msgs)
    if not (has_user and has_assistant):
        return None, None
    system_prompt = "\n\n".join(system_parts).strip()
    return system_prompt, msgs


def with_system_prompt(messages, system_prompt):
    """Prepend the conversation's system prompt if present."""
    non_sys = [m for m in messages if m["role"] != "system"]
    if system_prompt:
        return [{"role": "system", "content": system_prompt}] + non_sys
    return non_sys


def load_conversations(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        raw_text = f.read()
    lines = [line for line in raw_text.splitlines() if line.strip()]
    records, bad_lines = [], 0
    for line in lines:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            bad_lines += 1
    if not records:
        data = json.loads(raw_text)
        if isinstance(data, list) and data and isinstance(data[0], dict) and ("role" in data[0] or "from" in data[0]):
            records = [data]
        elif isinstance(data, list):
            records = data
        else:
            records = [data]
        bad_lines = 0

    conversations, skipped = [], 0
    for rec in records:
        raw = _extract_raw_messages(rec)
        if not raw:
            skipped += 1
            continue
        sys_prompt, msgs = normalize_messages(raw)
        if not msgs:
            skipped += 1
            continue
        conversations.append({
            "system": sys_prompt,
            "messages": msgs,
            "quality": _extract_quality(rec),
            "friend_key": _extract_friend_key(rec, sys_prompt),
        })
    print(f"Lines read: {len(lines)} | Invalid JSON: {bad_lines} | Skipped: {skipped} | Valid dialogues (weighted): {len(conversations)}")
    return conversations


conversations = load_conversations(DATA_PATH)
if len(conversations) == 0:
    raise RuntimeError("No valid dialogues found. Check the dataset file format.")

DEFAULT_SYSTEM_PROMPT = next((c["system"] for c in conversations if c["system"]), "")
n_turns = [sum(1 for m in c["messages"] if m["role"] == "assistant") for c in conversations]
print(f"Assistant replies: {sum(n_turns)} | Average per dialogue: {sum(n_turns) / len(n_turns):.1f}")
print(f"System prompt length in first sample: {len(DEFAULT_SYSTEM_PROMPT)} chars")
print("\nSample dialogue after normalization (with dataset system prompt):")
for m in with_system_prompt(conversations[0]["messages"], conversations[0]["system"])[:6]:
    print(f"  [{m['role']}] {m['content'][:150]!r}")

In [ ]:
# [Cell 5]: Load Qwen2.5-14B-Instruct (4-bit) via Unsloth + qwen-2.5 chat template
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

model_source = BASE_MODEL
drive_stage1_dir = os.path.join(DRIVE_DIR, "lora_stage1")
if RESUME_FROM_STAGE1:
    if os.path.isdir(STAGE1_LORA_DIR):
        model_source = STAGE1_LORA_DIR
        print("Resuming from local Stage 1 adapter:", STAGE1_LORA_DIR)
    elif os.path.isdir(drive_stage1_dir):
        model_source = drive_stage1_dir
        print("Resuming from Google Drive Stage 1 adapter:", drive_stage1_dir)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_source,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded successfully. VRAM reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
# [Cell 6]: Build Stage 1 and Stage 2 datasets (unshuffled per-friend validation split, ChatML formatting, quality filter)
from collections import defaultdict
from datasets import Dataset


def render(convo):
    return tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)


def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


TURN_OVERHEAD = 6  # <|im_start|>role\n ... <|im_end|>\n
_sys_token_cache = {}


def get_sys_tokens(system_prompt):
    if not system_prompt:
        return 0
    if system_prompt not in _sys_token_cache:
        _sys_token_cache[system_prompt] = count_tokens(render([{"role": "system", "content": system_prompt}]))
    return _sys_token_cache[system_prompt]


def to_exchanges(messages):
    """Group messages into [user, assistant] exchanges."""
    exchanges, current = [], []
    for m in messages:
        current.append(m)
        if m["role"] == "assistant":
            exchanges.append(current)
            current = []
    if current:
        exchanges.append(current)
    return exchanges


def chunk_conversation(messages, system_prompt, max_len):
    """Split long dialogues into chunks fitting within max_len tokens (with exchange overlap)."""
    sys_tokens = get_sys_tokens(system_prompt)
    exchanges = to_exchanges(messages)
    ex_lens = [sum(count_tokens(m["content"]) + TURN_OVERHEAD for m in ex) for ex in exchanges]
    chunks, cur, cur_len = [], [], sys_tokens
    for ex, ln in zip(exchanges, ex_lens):
        if cur and cur_len + ln > max_len:
            chunks.append(cur)
            overlap = cur[-CHUNK_OVERLAP_EXCHANGES:] if CHUNK_OVERLAP_EXCHANGES > 0 else []
            overlap_len = sum(o_len for _, o_len in overlap)
            if sys_tokens + overlap_len + ln > max_len:
                overlap, overlap_len = [], 0
            cur, cur_len = list(overlap), sys_tokens + overlap_len
        cur.append((ex, ln))
        cur_len += ln
    if cur:
        chunks.append(cur)
    result = []
    for ch in chunks:
        flat = [m for ex, _ in ch for m in ex]
        while flat and flat[-1]["role"] == "user":
            flat.pop()
        if any(m["role"] == "assistant" for m in flat) and any(m["role"] == "user" for m in flat):
            result.append(flat)
    return result


def build_examples(convs, max_len):
    examples = []
    for c in convs:
        recency = c.get("recency", 0.5)
        for chunk in chunk_conversation(c["messages"], c["system"], max_len):
            full = with_system_prompt(chunk, c["system"])
            text = render(full)
            examples.append({
                "text": text,
                "messages": full,
                "quality": c["quality"],
                "recency": recency,
                "n_tokens": count_tokens(text),
            })
    return examples


URL_RE = re.compile(r"https?://\S+")
LOW_INFO = {"ок", "ok", "окей", "ага", "угу", "да", "так", "ні", "нет", "no", "yes", "+", "лол", "lol", "хм", "ясно", "понял", "зрозумів"}


def quality_score(example):
    """
    Heuristic quality score of a dialogue chunk for Stage 2 annealing:
    - Rewards short, punchy replies (STAGE2_MIN_REPLY_CHARS..STAGE2_MAX_REPLY_CHARS, <= STAGE2_MAX_REPLY_LINES lines).
    - Penalizes rambling walls of text (> STAGE2_MAX_REPLY_CHARS or > STAGE2_MAX_REPLY_LINES lines).
    - Favors natural 2..6 exchange dialogues and strongly prioritizes recent (post-2024) chats.
    """
    replies = [m["content"] for m in example["messages"] if m["role"] == "assistant"]
    n = len(replies)
    if n == 0:
        return -1.0
    good = 0.0
    for r in replies:
        stripped = URL_RE.sub("", r).strip()
        if not stripped:
            continue
        num_lines = len(stripped.splitlines())
        if stripped.lower().strip(".,!?)(:; ") in LOW_INFO:
            good += 0.15
        elif len(stripped) > STAGE2_MAX_REPLY_CHARS or num_lines > STAGE2_MAX_REPLY_LINES:
            good -= 0.5
        elif STAGE2_MIN_REPLY_CHARS <= len(stripped) <= STAGE2_MAX_REPLY_CHARS:
            good += 1.0
        else:
            good += 0.3
    good_ratio = good / n
    uniqueness = len(set(r.strip().lower() for r in replies)) / n
    turns_bonus = 1.0 if 2 <= n <= 6 else (0.5 if n > 6 else 0.3)
    recency_bonus = example.get("recency", 0.5)
    score = 0.45 * good_ratio + 0.15 * uniqueness + 0.15 * turns_bonus + 0.25 * recency_bonus
    if example["quality"] is not None:
        score += example["quality"]
    return score


def split_train_val_per_friend(convs, val_fraction, min_convs):
    """
    Allocate val_fraction (5%) of dialogues from each friend/chat bucket to validation
    without pre-split shuffling so temporal duplicates and adjacent sliding-window chunks
    do not leak between train and validation.
    """
    if len(convs) < min_convs or val_fraction <= 0:
        for idx, c in enumerate(convs):
            c["recency"] = idx / max(1, len(convs) - 1)
        return convs[:], [], {}

    buckets = defaultdict(list)
    for c in convs:
        buckets[c["friend_key"]].append(c)

    train_convs, val_convs = [], []
    bucket_summary = {}

    for fkey, items in buckets.items():
        # Group identical weighted copies by dialogue content signature to prevent duplicate leakage
        unique_groups = []
        sig_to_group = {}
        for item in items:
            sig = tuple((m["role"], m["content"]) for m in item["messages"])
            if sig not in sig_to_group:
                group = []
                sig_to_group[sig] = group
                unique_groups.append(group)
            sig_to_group[sig].append(item)

        n_unique = len(unique_groups)
        for idx, g in enumerate(unique_groups):
            rec_val = idx / max(1, n_unique - 1)
            for item in g:
                item["recency"] = rec_val

        n_val_unique = int(round(n_unique * val_fraction)) if n_unique >= 20 else 0
        n_train_unique = n_unique - n_val_unique

        b_train, b_val = 0, 0
        for g in unique_groups[:n_train_unique]:
            train_convs.extend(g)
            b_train += len(g)
        for g in unique_groups[n_train_unique:]:
            # Keep one copy per unique validation dialogue for objective perplexity measurement
            val_convs.append(g[0])
            b_val += 1
        bucket_summary[fkey] = (b_train, b_val)

    return train_convs, val_convs, bucket_summary


train_convs, val_convs, bucket_summary = split_train_val_per_friend(conversations, VAL_FRACTION, MIN_CONVS_FOR_VAL)
print(f"Per-friend validation split (unshuffled, {VAL_FRACTION * 100:.0f}% per bucket): train={len(train_convs)}, val={len(val_convs)}")
for fkey, (b_tr, b_va) in bucket_summary.items():
    short_key = fkey[:70] + "..." if len(fkey) > 70 else fkey
    print(f"  Bucket: {short_key:<73} | train={b_tr:>5} | val={b_va:>4}")

# Stage 1 — full training set
stage1_examples = build_examples(train_convs, STAGE1_MAX_LEN)
stage1_val_examples = build_examples(val_convs, STAGE1_MAX_LEN) if val_convs else []

# Stage 2 — deduplicated top quality modern short-reply chunks
stage2_raw_pool = build_examples(train_convs, STAGE2_MAX_LEN)
stage2_pool = []
_seen_stage2_sigs = set()
for ex in stage2_raw_pool:
    sig = tuple((m["role"], m["content"]) for m in ex["messages"] if m["role"] != "system")
    if sig not in _seen_stage2_sigs:
        _seen_stage2_sigs.add(sig)
        ex["score"] = quality_score(ex)
        stage2_pool.append(ex)
stage2_pool.sort(key=lambda e: e["score"], reverse=True)
k = min(len(stage2_pool), max(STAGE2_MIN_EXAMPLES, int(len(stage2_pool) * STAGE2_TOP_FRACTION)))
stage2_examples = stage2_pool[:k]
random.seed(SEED)
random.shuffle(stage2_examples)
stage2_val_examples = build_examples(val_convs, STAGE2_MAX_LEN) if val_convs else []


def to_dataset(examples):
    return Dataset.from_list([{"text": e["text"]} for e in examples]) if examples else None


stage1_train_ds = to_dataset(stage1_examples)
stage1_val_ds = to_dataset(stage1_val_examples)
stage2_train_ds = to_dataset(stage2_examples)
stage2_val_ds = to_dataset(stage2_val_examples)


def _stats(name, examples):
    if not examples:
        print(f"{name:<22} 0 examples")
        return
    toks = sorted(e["n_tokens"] for e in examples)
    print(f"{name:<22} {len(examples):>6} examples | tokens: median={toks[len(toks) // 2]}, max={toks[-1]}, total={sum(toks):,}")


print()
_stats("Stage 1 train", stage1_examples)
_stats("Stage 1 val", stage1_val_examples)
_stats("Stage 2 train (top)", stage2_examples)
_stats("Stage 2 val", stage2_val_examples)
print(f"Stage 2 quality threshold: score >= {stage2_pool[k - 1]['score']:.3f}")
print("\nSample formatted text (ChatML / qwen-2.5):\n")
print(stage1_examples[0]["text"][:1200])

In [ ]:
# [Cell 7]: Attach LoRA adapter (r=32, alpha=48, target_modules=TARGET_MODULES)
if hasattr(model, "peft_config"):
    print("LoRA adapter is already attached (resumed from Stage 1).")
    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )
model.print_trainable_parameters()

In [ ]:
# [Cell 8]: Helper functions — trainer config, train_on_responses_only, inference, Google Drive backup
import gc
import math
import shutil
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer, DataCollatorForSeq2Seq

_SFT_PARAMS = set(inspect.signature(SFTConfig.__init__).parameters)


def make_sft_config(output_dir, max_len, lr, epochs, batch, grad_acc, warmup_ratio,
                    weight_decay, scheduler, n_train, has_eval):
    steps_per_epoch = max(1, math.ceil(n_train / (batch * grad_acc)))
    total_steps = steps_per_epoch * epochs
    eval_every = max(5, total_steps // 10)
    kw = dict(
        output_dir=output_dir,
        dataset_text_field="text",
        packing=False,
        dataset_num_proc=2,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=grad_acc,
        num_train_epochs=epochs,
        learning_rate=lr,
        lr_scheduler_type=scheduler,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        max_grad_norm=1.0,
        optim="adamw_8bit",
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        logging_steps=max(1, min(10, total_steps // 20)),
        save_strategy="steps",
        save_steps=max(50, total_steps // 4),
        save_total_limit=2,
        seed=SEED,
        report_to="none",
    )
    kw["max_length" if "max_length" in _SFT_PARAMS else "max_seq_length"] = max_len
    strategy_key = "eval_strategy" if "eval_strategy" in _SFT_PARAMS else "evaluation_strategy"
    if has_eval:
        kw[strategy_key] = "steps"
        kw["eval_steps"] = eval_every
        kw["per_device_eval_batch_size"] = batch
    else:
        kw[strategy_key] = "no"
    print(f"-> {output_dir}: {n_train} examples, {steps_per_epoch} steps/epoch, {total_steps} total steps, lr={lr}, scheduler={scheduler}")
    return SFTConfig(**kw)


def build_trainer(train_ds, eval_ds, args):
    common = dict(
        model=model,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=args,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    )
    try:
        trainer = SFTTrainer(processing_class=tokenizer, **common)
    except TypeError:
        trainer = SFTTrainer(tokenizer=tokenizer, **common)
    # Calculate loss strictly on assistant responses
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    return trainer


def show_masking(trainer, idx=0, max_chars=1500):
    """Display tokens where loss is active (system and user tokens are masked)."""
    sample = trainer.train_dataset[idx]
    pad_id = tokenizer.pad_token_id
    trained = [t if t != -100 else pad_id for t in sample["labels"]]
    decoded = tokenizer.decode(trained).replace(tokenizer.pad_token, "")
    print("--- Tokens with active loss (assistant responses only) ---")
    print(decoded[:max_chars])
    print("----------------------------------------------------------")


def gpu_mem(tag=""):
    print(f"[VRAM {tag}] reserved={torch.cuda.memory_reserved() / 1024**3:.2f} GB | "
          f"peak={torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")


def free_memory(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


def backup_to_drive(src_path):
    """Immediately back up an adapter directory or file to Google Drive."""
    if COPY_TO_DRIVE and os.path.exists(src_path) and os.path.ismount("/content/drive"):
        os.makedirs(DRIVE_DIR, exist_ok=True)
        dst = os.path.join(DRIVE_DIR, os.path.basename(src_path.rstrip("/")))
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src_path, dst)
        print(f"[Drive Backup] Copied to Google Drive: {dst}")


def generate_reply(user_message, history=None, system_prompt=None, stream=True):
    FastLanguageModel.for_inference(model)
    sys_p = DEFAULT_SYSTEM_PROMPT if system_prompt is None else system_prompt
    msgs = with_system_prompt((history or []) + [{"role": "user", "content": user_message}], sys_p)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH - GEN_MAX_NEW_TOKENS,
    ).to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            repetition_penalty=GEN_REPETITION_PENALTY,
            streamer=streamer,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_smoke_test(title):
    print("=" * 80)
    print(title)
    print("=" * 80)
    try:
        for p in TEST_PROMPTS:
            print(f"\n[User]: {p}\n[Assistant]: ", end="")
            generate_reply(p, stream=True)
    except Exception as e:
        print(f"\n[Warning] Error during smoke test (ignored to keep pipeline running): {e}")
    finally:
        FastLanguageModel.for_training(model)


print("Helper functions ready.")

In [ ]:
# [Cell 9]: STAGE 1 — Base & vocabulary acquisition (lr=1.0e-4, cosine, context 4096)
FastLanguageModel.for_training(model)
torch.cuda.reset_peak_memory_stats()

args_stage1 = make_sft_config(
    output_dir=STAGE1_OUT,
    max_len=STAGE1_MAX_LEN,
    lr=STAGE1_LR,
    epochs=STAGE1_EPOCHS,
    batch=STAGE1_BATCH,
    grad_acc=STAGE1_GRAD_ACC,
    warmup_ratio=STAGE1_WARMUP_RATIO,
    weight_decay=STAGE1_WEIGHT_DECAY,
    scheduler=STAGE1_SCHEDULER,
    n_train=len(stage1_train_ds),
    has_eval=stage1_val_ds is not None,
)
trainer1 = build_trainer(stage1_train_ds, stage1_val_ds, args_stage1)
show_masking(trainer1)

stats1 = trainer1.train()
print(f"\nStage 1 finished in {stats1.metrics['train_runtime'] / 60:.1f} min | train_loss={stats1.metrics['train_loss']:.4f}")
if stage1_val_ds is not None:
    print("Stage 1 Eval:", trainer1.evaluate())
gpu_mem("after Stage 1")

model.save_pretrained(STAGE1_LORA_DIR)
tokenizer.save_pretrained(STAGE1_LORA_DIR)
print("Stage 1 adapter saved to:", STAGE1_LORA_DIR)
backup_to_drive(STAGE1_LORA_DIR)

In [ ]:
# [Cell 10]: Inference smoke test after Stage 1
run_smoke_test("Responses after STAGE 1")

In [ ]:
# [Cell 11]: STAGE 2 — Style annealing & refinement (lr=2e-5, cosine, weight_decay=0.05, top dialogues)
free_memory("trainer1")
FastLanguageModel.for_training(model)
torch.cuda.reset_peak_memory_stats()

args_stage2 = make_sft_config(
    output_dir=STAGE2_OUT,
    max_len=STAGE2_MAX_LEN,
    lr=STAGE2_LR,
    epochs=STAGE2_EPOCHS,
    batch=STAGE2_BATCH,
    grad_acc=STAGE2_GRAD_ACC,
    warmup_ratio=STAGE2_WARMUP_RATIO,
    weight_decay=STAGE2_WEIGHT_DECAY,
    scheduler=STAGE2_SCHEDULER,
    n_train=len(stage2_train_ds),
    has_eval=stage2_val_ds is not None,
)
# Continue training on Stage 1 LoRA weights with a fresh optimizer and lower learning rate
trainer2 = build_trainer(stage2_train_ds, stage2_val_ds, args_stage2)
show_masking(trainer2)

stats2 = trainer2.train()
print(f"\nStage 2 finished in {stats2.metrics['train_runtime'] / 60:.1f} min | train_loss={stats2.metrics['train_loss']:.4f}")
if stage2_val_ds is not None:
    print("Stage 2 Eval:", trainer2.evaluate())
gpu_mem("after Stage 2")

model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)
print("Final adapter saved to:", FINAL_LORA_DIR)
backup_to_drive(FINAL_LORA_DIR)

In [ ]:
# [Cell 12]: Final inference test (single-turn prompts + multi-turn conversation)
run_smoke_test("Responses after STAGE 2 (Final)")

print("\n" + "=" * 80 + "\nMulti-turn Conversation Test\n" + "=" * 80)
try:
    history = []
    for user_msg in ["Йо, ти де пропав?", "Та норм, а шо там у тебе нового?", "Го завтра на каву?"]:
        print(f"\n[User]: {user_msg}\n[Assistant]: ", end="")
        reply = generate_reply(user_msg, history=history, stream=True)
        history += [{"role": "user", "content": user_msg}, {"role": "assistant", "content": reply}]
except Exception as e:
    print(f"\n[Warning] Error during multi-turn test: {e}")
finally:
    FastLanguageModel.for_training(model)

In [ ]:
# [Cell 13]: Export final HuggingFace PEFT model and back up to Google Drive
# Use tools/convert_lora_to_gguf_local.py locally on the exported lora_final folder to build the GGUF LoRA adapter.
free_memory("trainer1", "trainer2")

model.save_pretrained(FINAL_LORA_DIR)
tokenizer.save_pretrained(FINAL_LORA_DIR)
saved_files = sorted(os.listdir(FINAL_LORA_DIR))
total_mb = sum(
    os.path.getsize(os.path.join(FINAL_LORA_DIR, f))
    for f in saved_files
    if os.path.isfile(os.path.join(FINAL_LORA_DIR, f))
) / (1024 ** 2)
print(f"Final LoRA model exported to: {FINAL_LORA_DIR} ({total_mb:.1f} MB)")
print("Files:", saved_files)

if COPY_TO_DRIVE:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src_dir in [STAGE1_LORA_DIR, FINAL_LORA_DIR]:
        if os.path.exists(src_dir):
            dst_dir = os.path.join(DRIVE_DIR, os.path.basename(src_dir.rstrip("/")))
            shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
            print("Backed up to Google Drive:", dst_dir)
    print("Export complete. Download 'lora_final' from Google Drive and run tools/convert_lora_to_gguf_local.py locally.")